In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src.utils.config import *
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)

# 03 Feature engineering
`src/features/build_features.py` builds one row per customer as of 2026-06-30 (not today's date).

In [2]:
f = pd.read_csv(PROCESSED/'customer_features.csv'); print(f.shape); f.head(3).T

(8000, 41)


,0,1,2
customer_id,CUST000001,CUST000002,CUST000003
age,43,54,17
gender,Other,Female,Female
country,United States,France,Germany
acquisition_channel,Organic Search,Direct,Referral
customer_segment,Student,Corporate Bundle,Individual
customer_tenure_days,1522,188,2389
current_plan,Premium,Standard,Premium
plan_price,22.99,15.49,22.99
auto_renew,1,1,1


## Required features (PRD 9.2) and 11 extras discovered during EDA
Required: customer_tenure_days, avg_watch_time_per_session, monthly_watch_time_trend, days_since_last_activity, login_frequency, support_ticket_count, avg_customer_satisfaction, failed_payment_count/rate, plan_changes, engagement_score, customer_lifetime_value (+ churn_risk_score added after modelling).

Extras: `watch_trend_log_ratio`, `watch_min_last_3m`, `watch_min_prev_3m`, `failed_payments_90d`, `avg_completion_pct`, `distinct_titles`, `distinct_genres`, `distinct_devices`, `escalated_tickets`, `pending_tickets`, `billing_ticket_count`, `negative_feedback_count`, `avg_feedback_rating`, `payment_methods_used`.

In [3]:
from sklearn.metrics import roc_auc_score
rows = []
for c in f.select_dtypes('number').columns:
    if c in ('churned',): continue
    a = roc_auc_score(f.churned, f[c].fillna(f[c].median())); rows.append((c, round(max(a, 1-a), 3), 'higher = more churn' if a > .5 else 'higher = less churn'))
pd.DataFrame(rows, columns=['feature','univariate_AUC','direction']).sort_values('univariate_AUC', ascending=False).head(15)

,feature,univariate_AUC,direction
11,watch_min_last_3m,0.614,higher = less churn
10,monthly_watch_time_trend,0.603,higher = less churn
9,days_since_last_activity,0.600,higher = more churn
32,engagement_score,0.592,higher = less churn
13,watch_trend_log_ratio,0.584,higher = less churn
20,failed_payment_rate,0.580,higher = more churn
19,failed_payment_count,0.574,higher = more churn
23,customer_lifetime_value,0.552,higher = less churn
14,login_frequency,0.552,higher = less churn
25,avg_customer_satisfaction,0.550,higher = less churn


**Reading this honestly:** the single best feature reaches AUC 0.61. No feature is a strong predictor, which is why the model caps out near 0.66. The data generator added a lot of noise, as the dictionary warned.

In [4]:
print(f.isna().sum()[lambda s: s>0])  # NaNs are left for the model's imputer (median + missing-indicator), not hidden

avg_watch_time_per_session      20
avg_completion_pct              20
primary_device                  20
avg_customer_satisfaction     4460
avg_resolution_hours          4460
avg_feedback_rating           4439
dtype: int64
